In [1]:
import pandas as pd
import os
import glob
from tqdm import tqdm

# ==========================================
# 1. 설정: 경로 및 키워드 정의
# ==========================================
INPUT_FOLDER = r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\123_ticker"
OUTPUT_FOLDER = r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\123_ESG"

# 키워드 리스트 (필요에 따라 단어를 더 추가하거나 빼세요)
# 뉴스는 '명사' 위주로 매칭하는 것이 좋습니다.
keywords_E = [
    '탄소', '온실가스', '배출', '친환경', '에너지', '재생', '태양광', '풍력', '수소', 
    '기후', '환경', '오염', '폐기물', '리사이클', '재활용', '넷제로', 'RE100', 'ESG'
]

keywords_S = [
    '안전', '재해', '사망', '사고', '근로자', '노동', '임금', '고용', '채용', '노조', 
    '파업', '인권', '차별', '동반성장', '상생', '기부', '사회공헌', '개인정보', '갑질', '하청'
]

keywords_G = [
    '이사회', '주주', '배당', '지배구조', '거버넌스', '윤리', '횡령', '배임', '뇌물', 
    '불공정', '회계', '감사', '경영권', '승계', '사외이사', '의결권', '스튜어드십'
]

# ==========================================
# 2. 분류 함수
# ==========================================
def classify_by_keyword(text):
    if not isinstance(text, str):
        return "None"
    
    # 점수 계산 (키워드가 많이 등장한 쪽으로 분류)
    count_e = sum(text.count(word) for word in keywords_E)
    count_s = sum(text.count(word) for word in keywords_S)
    count_g = sum(text.count(word) for word in keywords_G)
    
    # 키워드가 하나도 없으면 None
    if count_e == 0 and count_s == 0 and count_g == 0:
        return "None"
    
    # 가장 많이 등장한 카테고리 선정
    counts = {'Environmental': count_e, 'Social': count_s, 'Governance': count_g}
    max_label = max(counts, key=counts.get)
    
    return max_label

# ==========================================
# 3. 실행 로직
# ==========================================
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

csv_files = glob.glob(os.path.join(INPUT_FOLDER, "*.csv"))
print(f"▶ 총 {len(csv_files)}개의 파일을 발견했습니다. 키워드 분석을 시작합니다.")

for file_path in tqdm(csv_files, desc="진행 중"):
    try:
        filename = os.path.basename(file_path)
        
        # 파일 읽기
        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='cp949')

        # 본문 컬럼 찾기
        target_col = None
        for col in ['본문', 'content', 'text', '기사내용']:
            if col in df.columns:
                target_col = col
                break
        
        if target_col is None:
            continue

        # 키워드 분류 적용 (apply 함수는 빠름)
        df['ESG_Label'] = df[target_col].astype(str).apply(classify_by_keyword)
        
        # 'None'이 아닌 데이터만 남길지 여부 (선택사항)
        # df = df[df['ESG_Label'] != 'None'] 

        # 저장
        save_path = os.path.join(OUTPUT_FOLDER, "keyword_" + filename)
        df.to_csv(save_path, index=False, encoding='utf-8-sig')

    except Exception as e:
        print(f"Error in {file_path}: {e}")

print(f"\n▶ 완료! 결과 파일들이 {OUTPUT_FOLDER}에 저장되었습니다.")

▶ 총 123개의 파일을 발견했습니다. 키워드 분석을 시작합니다.


진행 중: 100%|██████████| 123/123 [00:05<00:00, 23.38it/s]


▶ 완료! 결과 파일들이 C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\123_ESG에 저장되었습니다.


In [2]:
import pandas as pd
import os
import glob

# ==========================================
# [설정] 분석 결과가 저장된 폴더 경로
# ==========================================
# 기존에 분석 돌리셨던 결과 폴더 경로를 넣어주세요.
RESULT_FOLDER = r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\123_ESG"

# ==========================================
# [실행] 집계 및 화면 출력
# ==========================================
csv_files = glob.glob(os.path.join(RESULT_FOLDER, "*.csv"))

if not csv_files:
    print("❌ 경로를 확인해주세요. 해당 폴더에 파일이 없습니다.")
else:
    print(f"\n▶ 총 {len(csv_files)}개 파일의 분석 결과를 집계합니다...\n")
    
    summary_list = []
    
    for file_path in csv_files:
        try:
            # 1. 파일 읽기
            # "None"을 NaN으로 인식하지 않도록 설정 (keep_default_na=False)
            df = pd.read_csv(file_path, keep_default_na=False, na_values=['', '#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan', '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NA', 'NULL', 'NaN', 'n/a', 'nan', 'null'])
            
            # 파일명 깔끔하게 정리
            file_name = os.path.basename(file_path).replace("keyword_", "").replace("esg_", "").replace(".csv", "")
            
            if 'ESG_Label' in df.columns:
                # 2. 개수 세기
                total_count = len(df)
                counts = df['ESG_Label'].value_counts()
                
                count_e = counts.get('Environmental', 0)
                count_s = counts.get('Social', 0)
                count_g = counts.get('Governance', 0)
                
                # [중요] None 개수는 전체에서 나머지를 빼서 계산 (가장 정확함)
                # 혹은 데이터프레임에서 "None" 문자열 개수 직접 확인
                count_none = total_count - (count_e + count_s + count_g)
                
                # 데이터 저장
                summary_list.append({
                    '기업명': file_name,
                    '전체': total_count,
                    'E(환경)': count_e,
                    'S(사회)': count_s,
                    'G(지배구조)': count_g,
                    'None': count_none
                })
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue

    # 3. 결과 출력
    if summary_list:
        result_df = pd.DataFrame(summary_list)
        
        # 기사 많은 순서로 정렬
        result_df = result_df.sort_values(by='전체', ascending=False)
        
        # 화면 출력 옵션 (짤림 방지)
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        
        print("-" * 90)
        print(result_df.to_string(index=False)) 
        print("-" * 90)
        
        # 전체 합계 출력
        print("\n[ 전체 데이터 합계 ]")
        total_sum = result_df[['전체', 'E(환경)', 'S(사회)', 'G(지배구조)', 'None']].sum()
        print(total_sum.to_string())
        
        # (선택사항) 최종 결과를 엑셀로 저장하고 싶으시면 아래 주석을 푸세요
        # result_df.to_csv(os.path.join(RESULT_FOLDER, "final_summary_fixed.csv"), index=False, encoding='utf-8-sig')
        
    else:
        print("집계할 데이터가 없습니다.")


▶ 총 123개 파일의 분석 결과를 집계합니다...

------------------------------------------------------------------------------------------
                               기업명    전체  E(환경)  S(사회)  G(지배구조)  None
    롯데하이마트_predicted_results_hope2 21397   1178   1787     1220 17212
      우리은행_predicted_results_hope2 15636    332   1571     1694 12039
      롯데푸드_predicted_results_hope2 14156    647   1174     2115 10220
       이노션_predicted_results_hope2 11187    543    731      822  9091
     DB하이텍_predicted_results_hope2  9654    353    484      991  7826
      남양유업_predicted_results_hope2  7687    270   1055     1546  4816
       HDC_predicted_results_hope2  5595    213    264      452  4666
    SGC에너지_predicted_results_hope2  4622   1509    174      524  2415
     HJ중공업_predicted_results_hope2  4470    372    411      323  3364
    아시아나항공_predicted_results_hope2  4170    141    204      552  3273
 HDC현대산업개발_predicted_results_hope2  3740    179    441      244  2876
     HL홀딩스_predicted_results_hope2  36

In [ ]:
from pathlib import Path
import pandas as pd

# =========================
# 1) 설정 (여기만 수정)
# =========================
DATA_DIR = Path(r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\123_ESG")   # <- 입력 폴더
LABEL_COL = "ESG_Label"                  # <- ESG/NONE 들어있는 컬럼명으로 수정 (예: "esg_label", "category", "ESG" 등)

# 출력 폴더: ESG_123 폴더와 같은 상위(부모) 위치에 생성
BASE_DIR = DATA_DIR.parent
OUT_ESG = BASE_DIR / "ESG분류완료_123"
OUT_NONE = BASE_DIR / "ESGNone_123"

OUT_ESG.mkdir(parents=True, exist_ok=True)
OUT_NONE.mkdir(parents=True, exist_ok=True)

# =========================
# 2) 처리
# =========================
csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"CSV 파일을 못 찾았어요: {DATA_DIR}")

for fp in csv_files:
    # 인코딩 대응
    try:
        df = pd.read_csv(fp, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(fp, encoding="cp949")

    if LABEL_COL not in df.columns:
        print(f"[스킵] {fp.name}: '{LABEL_COL}' 컬럼이 없음. (컬럼 예시: {list(df.columns)[:10]})")
        continue

    # 라벨 정리
    s = df[LABEL_COL].astype(str).str.strip().str.upper()

    # ESG / NONE(결측 포함) 분리
    esg_mask = s.eq("ESG")
    none_mask = s.eq("NONE") | df[LABEL_COL].isna() | s.eq("NAN") | s.eq("")

    df_esg = df.loc[esg_mask].copy()
    df_none = df.loc[none_mask].copy()

    # 원본 파일명 그대로 저장 (원하면 _ESG/_NONE 붙이도록 변경 가능)
    out_esg_path = OUT_ESG / fp.name
    out_none_path = OUT_NONE / fp.name

    # 행이 0개여도 파일 저장(원하면 0개면 저장 안 하도록 옵션 줄 수 있음)
    df_esg.to_csv(out_esg_path, index=False, encoding="utf-8-sig")
    df_none.to_csv(out_none_path, index=False, encoding="utf-8-sig")

    print(f"[완료] {fp.name} -> ESG:{len(df_esg):,}행 저장, NONE:{len(df_none):,}행 저장")

print("전체 분류 저장 완료!")


In [ ]:
from pathlib import Path
import pandas as pd

# =========================
# 1) 설정 (여기만 수정)
# =========================
DATA_DIR = Path(r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\123_ESG")   # <- ESG_123 폴더 경로로 수정
LABEL_COL = "ESG_Label"                  # <- 라벨 컬럼명(예: "ESG", "esg_label", "category", "label" 등)

# =========================
# 2) 출력 폴더 (ESG_123 밖에 만들기)
# =========================
BASE_DIR = DATA_DIR.parent
OUT_ESG  = BASE_DIR / "ESG분류완료_123"
OUT_NONE = BASE_DIR / "ESGNone_123"
OUT_ESG.mkdir(parents=True, exist_ok=True)
OUT_NONE.mkdir(parents=True, exist_ok=True)

# =========================
# 3) ESG / NONE 분리 규칙
#    - ESG: Environmental / Social / Governance / (E:, S:, G: 등)
#    - NONE: None/NaN/빈값
# =========================
ESG_REGEX = r"(environment(al)?|social|governance|govern)|(^[esg]\s*[:\-])|(^[esg]$)"

NONE_TOKENS = {"none", "nan", "null", ""}

csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"CSV 파일을 못 찾았어요: {DATA_DIR}")

for fp in csv_files:
    # 인코딩 대응 (utf-8-sig 우선, 안 되면 cp949)
    try:
        df = pd.read_csv(fp, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(fp, encoding="cp949")

    if LABEL_COL not in df.columns:
        print(f"[스킵] {fp.name}: '{LABEL_COL}' 컬럼 없음 (컬럼 예시: {list(df.columns)[:10]})")
        continue

    s = df[LABEL_COL].astype("string").str.strip().str.lower()

    none_mask = s.isna() | s.isin(list(NONE_TOKENS))
    esg_mask  = s.str.contains(ESG_REGEX, regex=True, na=False)

    df_esg  = df.loc[esg_mask].copy()
    df_none = df.loc[none_mask].copy()

    # 원본 파일명 그대로 저장 (원하면 _ESG/_NONE 붙일 수 있음)
    df_esg.to_csv(OUT_ESG / fp.name, index=False, encoding="utf-8-sig")
    df_none.to_csv(OUT_NONE / fp.name, index=False, encoding="utf-8-sig")

    print(f"[완료] {fp.name} -> ESG:{len(df_esg):,}행 / NONE:{len(df_none):,}행")

print("전체 분류 저장 완료!")


In [34]:
from pathlib import Path

folder = Path(r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\ESG분류완료_123")   # <- 여기만 바꿔
csv_files = list(folder.glob("*.csv"))

print(len(csv_files))
print(csv_files[:5])  # 앞 5개 확인


123
[WindowsPath('C:/Users/shcho/OneDrive/바탕 화면/최종 프로젝트/추가자료/코드/ESG분류완료_123/keyword_AK홀딩스_predicted_results_hope2.csv'), WindowsPath('C:/Users/shcho/OneDrive/바탕 화면/최종 프로젝트/추가자료/코드/ESG분류완료_123/keyword_BGF_predicted_results_hope2.csv'), WindowsPath('C:/Users/shcho/OneDrive/바탕 화면/최종 프로젝트/추가자료/코드/ESG분류완료_123/keyword_CJ CGV_predicted_results_hope2.csv'), WindowsPath('C:/Users/shcho/OneDrive/바탕 화면/최종 프로젝트/추가자료/코드/ESG분류완료_123/keyword_CR홀딩스_predicted_results_hope2.csv'), WindowsPath('C:/Users/shcho/OneDrive/바탕 화면/최종 프로젝트/추가자료/코드/ESG분류완료_123/keyword_DB하이텍_predicted_results_hope2.csv')]


In [31]:
from pathlib import Path
import pandas as pd

folder = Path(r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\ESG분류완료_123")   # <- 여기만 바꿔

for fp in folder.glob("*.csv"):
    df = pd.read_csv(fp, encoding="utf-8-sig")  # 필요하면 cp949로 바꿔
    print(fp.name, df.shape)


keyword_AK홀딩스_predicted_results_hope2.csv (57, 10)
keyword_BGF_predicted_results_hope2.csv (331, 10)
keyword_CJ CGV_predicted_results_hope2.csv (177, 10)
keyword_CR홀딩스_predicted_results_hope2.csv (719, 10)
keyword_DB하이텍_predicted_results_hope2.csv (1828, 10)
keyword_DN오토모티브_predicted_results_hope2.csv (129, 10)
keyword_F&F홀딩스_predicted_results_hope2.csv (4, 10)
keyword_HDC_predicted_results_hope2.csv (929, 10)
keyword_HDC현대산업개발_predicted_results_hope2.csv (864, 10)
keyword_HJ중공업_predicted_results_hope2.csv (1106, 10)
keyword_HL홀딩스_predicted_results_hope2.csv (982, 10)
keyword_JW중외제약_predicted_results_hope2.csv (18, 10)
keyword_JW홀딩스_predicted_results_hope2.csv (57, 10)
keyword_KG스틸_predicted_results_hope2.csv (28, 10)
keyword_LF_predicted_results_hope2.csv (66, 10)
keyword_LG생명과학_predicted_results_hope2.csv (68, 10)
keyword_LX하우시스_predicted_results_hope2.csv (1, 10)
keyword_LX홀딩스_predicted_results_hope2.csv (52, 10)
keyword_SBS_predicted_results_hope2.csv (192, 10)
keyword_SGC에너지_predi

In [32]:
from pathlib import Path
import pandas as pd
import re

# ✅ 여기만 바꿔줘
INPUT_DIR = Path(r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\ESG분류완료_123")
OUT_DIR = INPUT_DIR.parent / f"{INPUT_DIR.name}_4cols_only"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# (혹시 컬럼명이 살짝 다를 때 대비용 후보)
DATE_CANDS   = ["일자", "date", "날짜"]
TICKER_CANDS = ["ticker", "종목코드", "code", "티커"]
PRED_CANDS   = ["pred_label", "pred", "prediction"]
ESG_CANDS    = ["ESG_Label", "ESG_label", "esg_label", "label", "ESG"]

def pick_col(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

def pad_ticker(x):
    """5930 / 005930 / 005930.0 -> 6자리 0패딩 '005930'"""
    if pd.isna(x):
        return x
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    digits = re.sub(r"\D", "", s)
    if digits and len(digits) <= 6:
        return digits.zfill(6)
    return digits if digits else s

csv_files = sorted(INPUT_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"CSV 파일이 없어요: {INPUT_DIR}")

for fp in csv_files:
    # ✅ 날짜 안 바꾸려면 dtype=str(또는 string)로 읽는 게 핵심!
    try:
        df = pd.read_csv(fp, encoding="utf-8-sig", dtype=str)
    except UnicodeDecodeError:
        df = pd.read_csv(fp, encoding="cp949", dtype=str)

    date_col = pick_col(df, DATE_CANDS)
    tick_col = pick_col(df, TICKER_CANDS)
    pred_col = pick_col(df, PRED_CANDS)
    esg_col  = pick_col(df, ESG_CANDS)

    missing = []
    if date_col is None: missing.append("일자")
    if tick_col is None: missing.append("ticker")
    if pred_col is None: missing.append("pred_label")
    if esg_col is None:  missing.append("ESG_Label")

    if missing:
        print(f"[스킵] {fp.name}: 컬럼 못 찾음 -> {missing} | 현재 컬럼 일부: {list(df.columns)[:15]}")
        continue

    out = df[[date_col, tick_col, pred_col, esg_col]].copy()
    out.columns = ["일자", "ticker", "pred_label", "ESG_Label"]

    # ✅ ticker만 0패딩 (날짜는 건드리지 않음)
    out["ticker"] = out["ticker"].apply(pad_ticker)

    out.to_csv(OUT_DIR / fp.name, index=False, encoding="utf-8-sig")
    print(f"[완료] {fp.name} -> 저장")

print(f"✅ 끝! 저장 폴더: {OUT_DIR}")


[완료] keyword_AK홀딩스_predicted_results_hope2.csv -> 저장
[완료] keyword_BGF_predicted_results_hope2.csv -> 저장
[완료] keyword_CJ CGV_predicted_results_hope2.csv -> 저장
[완료] keyword_CR홀딩스_predicted_results_hope2.csv -> 저장
[완료] keyword_DB하이텍_predicted_results_hope2.csv -> 저장
[완료] keyword_DN오토모티브_predicted_results_hope2.csv -> 저장
[완료] keyword_F&F홀딩스_predicted_results_hope2.csv -> 저장
[완료] keyword_HDC_predicted_results_hope2.csv -> 저장
[완료] keyword_HDC현대산업개발_predicted_results_hope2.csv -> 저장
[완료] keyword_HJ중공업_predicted_results_hope2.csv -> 저장
[완료] keyword_HL홀딩스_predicted_results_hope2.csv -> 저장
[완료] keyword_JW중외제약_predicted_results_hope2.csv -> 저장
[완료] keyword_JW홀딩스_predicted_results_hope2.csv -> 저장
[완료] keyword_KG스틸_predicted_results_hope2.csv -> 저장
[완료] keyword_LF_predicted_results_hope2.csv -> 저장
[완료] keyword_LG생명과학_predicted_results_hope2.csv -> 저장
[완료] keyword_LX하우시스_predicted_results_hope2.csv -> 저장
[완료] keyword_LX홀딩스_predicted_results_hope2.csv -> 저장
[완료] keyword_SBS_predicted_results_hope2.cs

In [ ]:
# from pathlib import Path
# import pandas as pd
# from tqdm import tqdm

# # ================
# # 설정 (여기만 수정)
# # ================
# INPUT_DIR = Path(r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\ESG분류완료_123")   # ESG 기사 파일들 있는 폴더
# OUT_DIR   = INPUT_DIR.parent / "ESG분류완료_123_sentiment"
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# # (선택) 감성 라벨 컬럼이 이미 있다면 지정 (없으면 자동 추론/모델 사용)
# # 예: "sentiment", "감성", "호재악재", "sentiment_label" 등
# SENTIMENT_COL_CANDIDATES = ["sentiment", "sentiment_label", "감성", "감성라벨", "호재악재"]

# # 텍스트 컬럼 후보(제목/본문)
# TITLE_CANDIDATES = ["제목", "title"]
# BODY_CANDIDATES  = ["본문", "content", "body", "기사본문"]

# # 날짜/티커 컬럼 후보(일별 집계용)
# DATE_CANDIDATES  = ["일자", "date", "날짜"]
# TICKER_CANDIDATES = ["ticker", "종목코드", "code"]

# # =======================
# # 1) 유틸: 컬럼 자동 찾기
# # =======================
# def pick_col(df, candidates):
#     for c in candidates:
#         if c in df.columns:
#             return c
#     return None

# # ============================
# # 2) 유틸: 라벨 -> 점수 매핑
# # ============================
# def map_label_to_score(x: str):
#     """
#     자주 쓰는 표기들을 최대한 커버:
#     - '호재(긍정)', '긍정' -> +1
#     - '악재(부정)', '부정' -> -1
#     - '중립' -> 0
#     - 'positive/negative/neutral' 등도 처리
#     """
#     if pd.isna(x):
#         return None
#     s = str(x).strip().lower()

#     if any(k in s for k in ["호재", "긍정", "positive", "pos"]):
#         return 1
#     if any(k in s for k in ["악재", "부정", "negative", "neg"]):
#         return -1
#     if any(k in s for k in ["중립", "neutral", "neu"]):
#         return 0

#     # 혹시 이미 숫자로 들어온 경우
#     try:
#         v = float(s)
#         if v in [-1, 0, 1]:
#             return int(v)
#     except:
#         pass

#     return None

# # =========================================
# # 3) (선택) 모델로 감성 분류해서 점수 생성
# # =========================================
# USE_MODEL_IF_NO_LABEL = True

# MODEL_NAME_OR_PATH = r"C:\경로\sentiment_model"  # <- 너희 로컬 파인튜닝 모델 폴더 or HF 모델명
# BATCH_SIZE = 32

# def build_sentiment_pipeline():
#     """
#     감성 라벨 컬럼이 없을 때만 사용됨.
#     너희가 쓰는 모델(로컬 경로/모델명)로 바꾸면 됨.
#     """
#     from transformers import pipeline
#     import torch

#     device = 0 if torch.cuda.is_available() else -1
#     clf = pipeline(
#         "text-classification",
#         model=MODEL_NAME_OR_PATH,
#         tokenizer=MODEL_NAME_OR_PATH,
#         device=device,
#         return_all_scores=True,
#         truncation=True,
#         max_length=256,
#     )
#     return clf

# def probs_to_score(scores_list):
#     """
#     return_all_scores=True 결과(라벨별 확률)를 받아
#     sentiment_score = P(pos) - P(neg) 로 계산 (연속값 -1~+1 근처)
#     라벨명이 모델마다 달라서 최대한 유연하게 처리.
#     """
#     # scores_list: [{"label": "...", "score": 0.7}, ...]
#     probs = {d["label"].lower(): float(d["score"]) for d in scores_list}

#     def find_prob(keys):
#         for k in keys:
#             for label, p in probs.items():
#                 if k in label:
#                     return p
#         return 0.0

#     p_pos = find_prob(["pos", "positive", "긍정"])
#     p_neg = find_prob(["neg", "negative", "부정"])
#     p_neu = find_prob(["neu", "neutral", "중립"])

#     score = p_pos - p_neg
#     return score, p_pos, p_neg, p_neu

# # ============================
# # 4) 메인 처리: 파일별 점수
# # ============================
# csv_files = sorted(INPUT_DIR.glob("*.csv"))
# if not csv_files:
#     raise FileNotFoundError(f"CSV가 없어요: {INPUT_DIR}")

# sent_clf = None  # 필요할 때만 생성

# daily_rows = []

# for fp in tqdm(csv_files, desc="Scoring sentiment"):
#     # 인코딩 대응
#     try:
#         df = pd.read_csv(fp, encoding="utf-8-sig")
#     except UnicodeDecodeError:
#         df = pd.read_csv(fp, encoding="cp949")

#     # 감성 라벨 컬럼 찾기(있으면 라벨 기반으로 점수)
#     sent_col = pick_col(df, SENTIMENT_COL_CANDIDATES)

#     # 텍스트 컬럼 찾기(모델 필요 시)
#     title_col = pick_col(df, TITLE_CANDIDATES)
#     body_col  = pick_col(df, BODY_CANDIDATES)

#     # 날짜/티커 컬럼 찾기(일별 집계)
#     date_col  = pick_col(df, DATE_CANDIDATES)
#     tick_col  = pick_col(df, TICKER_CANDIDATES)

#     # -------------------------
#     # (A) 라벨 컬럼이 있는 경우
#     # -------------------------
#     if sent_col is not None:
#         df["sentiment_score"] = df[sent_col].apply(map_label_to_score)

#     # -------------------------
#     # (B) 라벨 컬럼이 없으면 모델로 생성
#     # -------------------------
#     else:
#         if not USE_MODEL_IF_NO_LABEL:
#             print(f"[스킵] {fp.name}: 감성 라벨 컬럼 없음 & 모델 사용 OFF")
#             continue

#         if title_col is None and body_col is None:
#             print(f"[스킵] {fp.name}: 텍스트 컬럼(제목/본문) 없음")
#             continue

#         # 텍스트 만들기: 제목 + 본문(가능하면 둘 다)
#         t = ""
#         if title_col is not None:
#             t = df[title_col].fillna("").astype(str)
#         b = ""
#         if body_col is not None:
#             b = df[body_col].fillna("").astype(str)

#         text_series = (t + " " + b).str.strip()
#         texts = text_series.tolist()

#         if sent_clf is None:
#             sent_clf = build_sentiment_pipeline()

#         out_scores = []
#         p_pos_list, p_neg_list, p_neu_list = [], [], []

#         for i in range(0, len(texts), BATCH_SIZE):
#             batch = texts[i:i+BATCH_SIZE]
#             preds = sent_clf(batch)  # 각 원소가 label별 확률 리스트
#             for scores_list in preds:
#                 score, p_pos, p_neg, p_neu = probs_to_score(scores_list)
#                 out_scores.append(score)
#                 p_pos_list.append(p_pos)
#                 p_neg_list.append(p_neg)
#                 p_neu_list.append(p_neu)

#         df["sentiment_score"] = out_scores
#         df["p_pos"] = p_pos_list
#         df["p_neg"] = p_neg_list
#         df["p_neu"] = p_neu_list

#     # 저장(기사 단위)
#     out_path = OUT_DIR / fp.name
#     df.to_csv(out_path, index=False, encoding="utf-8-sig")

#     # -------------------------
#     # 일별 집계(가능할 때만)
#     # -------------------------
#     if date_col is not None and tick_col is not None and "sentiment_score" in df.columns:
#         tmp = df[[date_col, tick_col, "sentiment_score"]].copy()
#         tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")

#         daily = (
#             tmp.dropna(subset=[date_col, tick_col, "sentiment_score"])
#                .groupby([tick_col, date_col], as_index=False)
#                .agg(
#                    news_cnt=("sentiment_score", "size"),
#                    news_sentiment=("sentiment_score", "mean")
#                )
#         )
#         daily_rows.append(daily)

# # 일별 sentiment 저장(전 파일 합쳐서)
# if daily_rows:
#     daily_all = pd.concat(daily_rows, ignore_index=True)
#     daily_all.to_csv(OUT_DIR.parent / "daily_sentiment.csv", index=False, encoding="utf-8-sig")
#     print("✅ daily_sentiment.csv 저장 완료")
# else:
#     print("⚠️ 날짜/티커 컬럼이 없어서 일별 집계는 건너뜀")

# print(f"✅ 기사별 sentiment 파일 저장 완료: {OUT_DIR}")


Scoring sentiment: 100%|██████████| 123/123 [00:01<00:00, 70.27it/s]


✅ daily_sentiment.csv 저장 완료
✅ 기사별 sentiment 파일 저장 완료: C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\ESG분류완료_123_sentiment


In [ ]:
from pathlib import Path
import pandas as pd
import re
from collections import Counter

# =========================
# 0) 경로 설정 (여기만 수정)
# =========================
INPUT_DIR = Path(r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\ESG분류완료_123")  # <- 종목별 csv들이 있는 폴더
OUT_DIR = INPUT_DIR.parent / "ESG_ticker분류"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 1) 컬럼 후보(자동 탐지)
# =========================
DATE_CANDS   = ["일자", "date", "날짜", "pubDate"]
TICKER_CANDS = ["ticker", "종목코드", "code", "티커"]
LABEL_CANDS  = ["ESG_Label", "esg", "esg_label", "category", "ESG", "분류", "라벨"]

def pick_col(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

def normalize_ticker(x):
    """예: 5930 / 005930 / 005930.0 -> '005930'"""
    if pd.isna(x):
        return None
    s = str(x).strip().replace(".0", "")
    digits = re.sub(r"\D", "", s)
    if digits:
        return digits.zfill(6) if len(digits) <= 6 else digits
    return s

def infer_rep_ticker(df, ticker_col, fp_stem):
    """대표 ticker: ticker 컬럼 최빈값 -> 파일명 6자리 -> stem"""
    rep = None
    if ticker_col and ticker_col in df.columns:
        ticks = df[ticker_col].dropna().map(normalize_ticker).tolist()
        if ticks:
            rep = Counter(ticks).most_common(1)[0][0]
    if rep is None:
        m = re.search(r"(\d{6})", fp_stem)
        rep = m.group(1) if m else fp_stem
    return rep

def classify_esg_label(x):
    """
    label 값이 Environmental/Social/Governance (또는 E:/S:/G: 표기)면
    E / S / G / None 반환
    """
    if pd.isna(x):
        return None
    s = str(x).strip().lower()

    if s in {"none", "nan", "null", ""}:
        return None

    # Environmental
    if "environment" in s or "환경" in s or re.match(r"^e\s*[:\-]", s) or s == "e":
        return "E"
    # Social
    if "social" in s or "사회" in s or re.match(r"^s\s*[:\-]", s) or s == "s":
        return "S"
    # Governance
    if "govern" in s or "지배구조" in s or "거버넌스" in s or re.match(r"^g\s*[:\-]", s) or s == "g":
        return "G"

    return None

# =========================
# 2) 처리
# =========================
csv_files = sorted(INPUT_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"CSV 파일이 없어요: {INPUT_DIR}")

for fp in csv_files:
    # 인코딩 대응
    try:
        df = pd.read_csv(fp, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(fp, encoding="cp949")

    date_col  = pick_col(df, DATE_CANDS)
    tick_col  = pick_col(df, TICKER_CANDS)
    label_col = pick_col(df, LABEL_CANDS)

    if label_col is None:
        print(f"[스킵] {fp.name}: label/분류 컬럼을 못 찾았어요. (컬럼 일부: {list(df.columns)[:12]})")
        continue

    # --- ticker 컬럼(표준)
    if tick_col is not None:
        df["ticker"] = df[tick_col].map(normalize_ticker)
    else:
        df["ticker"] = None

    # --- 일자 컬럼(표준)
    if date_col is not None:
        df["일자"] = pd.to_datetime(df[date_col], errors="coerce").dt.strftime("%Y-%m-%d")
    else:
        df["일자"] = None

    # --- label 컬럼(표준)
    df["label"] = df[label_col].astype("string").str.strip()

    # --- E/S/G 분류(원핫)
    cls = df["label"].apply(classify_esg_label)
    df["E"] = (cls == "E").astype(int)
    df["S"] = (cls == "S").astype(int)
    df["G"] = (cls == "G").astype(int)

    # 보기 좋게 label 표준화(원하면 주석처리 가능)
    df.loc[df["E"] == 1, "label"] = "Environmental"
    df.loc[df["S"] == 1, "label"] = "Social"
    df.loc[df["G"] == 1, "label"] = "Governance"

    # 대표 ticker 뽑아서 파일명 만들기
    rep_ticker = infer_rep_ticker(df, "ticker", fp.stem)

    # 컬럼 순서 정리: 요청한 컬럼들을 앞으로
    front = ["일자", "ticker", "label", "E", "S", "G"]
    rest = [c for c in df.columns if c not in front]
    df = df[front + rest]

    # 저장: ticker_ESG.csv
    out_path = OUT_DIR / f"{rep_ticker}_ESG.csv"
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"[완료] {fp.name} -> {out_path.name} | E:{df['E'].sum():,}, S:{df['S'].sum():,}, G:{df['G'].sum():,}")

print(f"✅ 끝! 결과 폴더: {OUT_DIR}")


[완료] keyword_AK홀딩스_predicted_results_hope2.csv -> 006840_ESG.csv | E:4, S:0, G:53
[완료] keyword_BGF_predicted_results_hope2.csv -> 027410_ESG.csv | E:110, S:96, G:125
[완료] keyword_CJ CGV_predicted_results_hope2.csv -> 079160_ESG.csv | E:38, S:55, G:84
[완료] keyword_CR홀딩스_predicted_results_hope2.csv -> 000480_ESG.csv | E:151, S:232, G:336
[완료] keyword_DB하이텍_predicted_results_hope2.csv -> 000990_ESG.csv | E:353, S:484, G:991
[완료] keyword_DN오토모티브_predicted_results_hope2.csv -> 007340_ESG.csv | E:19, S:55, G:55
[완료] keyword_F&F홀딩스_predicted_results_hope2.csv -> 007700_ESG.csv | E:0, S:0, G:4
[완료] keyword_HDC_predicted_results_hope2.csv -> 012630_ESG.csv | E:213, S:264, G:452
[완료] keyword_HDC현대산업개발_predicted_results_hope2.csv -> 294870_ESG.csv | E:179, S:441, G:244
[완료] keyword_HJ중공업_predicted_results_hope2.csv -> 097230_ESG.csv | E:372, S:411, G:323
[완료] keyword_HL홀딩스_predicted_results_hope2.csv -> 060980_ESG.csv | E:162, S:169, G:651
[완료] keyword_JW중외제약_predicted_results_hope2.csv -> 001060

In [26]:
s = df[date_col]

# 1) 먼저 문자열로 보고 8자리(YYYYMMDD) 파싱 시도
s_str = s.astype("string").str.strip()
d1 = pd.to_datetime(s_str, format="%Y%m%d", errors="coerce")

# 2) 나머지는 일반 파싱(YYYY-MM-DD, YYYY.MM.DD 등)
d2 = pd.to_datetime(s_str, errors="coerce")

# 3) 그래도 NaT인데 숫자면 엑셀 시리얼로 파싱
mask = d1.isna() & d2.isna() & s_str.str.fullmatch(r"\d+(\.0)?", na=False)
d3 = pd.to_datetime(pd.to_numeric(s_str.where(mask).str.replace(".0","", regex=False), errors="coerce"),
                    unit="D", origin="1899-12-30", errors="coerce")

df["일자"] = d1.fillna(d2).fillna(d3)


In [27]:
df["일자"] = df["일자"].dt.strftime("%Y-%m-%d")


In [28]:
print(df[date_col].head(5))
print(df["일자"].head(5))
print(df["일자"].dtype)


0    1970-01-01
1    1970-01-01
2    1970-01-01
3    1970-01-01
4    1970-01-01
Name: 일자, dtype: object
0    1970-01-01
1    1970-01-01
2    1970-01-01
3    1970-01-01
4    1970-01-01
Name: 일자, dtype: object
object


In [29]:
import pandas as pd

In [10]:
df = pd.read_csv('./ESG_ticker분류/115390_ESG.csv')

In [ ]:
# ticker에 0 추가 
from pathlib import Path
import pandas as pd
import re

# =========================
# 1) 경로 설정 (여기만 수정)
# =========================
INPUT_DIR = Path(r"C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\ESG_ticker분류")   # <- 입력 폴더
OUT_DIR = INPUT_DIR.parent / f"{INPUT_DIR.name}_4cols_pad"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 2) 컬럼명 후보
# =========================
DATE_CANDS   = ["일자", "date", "날짜"]
TICKER_CANDS = ["ticker", "종목코드", "code", "티커"]
PRED_CANDS   = ["pred_label", "pred", "prediction", "sentiment_pred", "감성예측", "감성라벨"]
ESG_CANDS    = ["ESG_Label", "ESG_label", "esg_label", "label", "ESG", "category", "분류"]

def pick_col(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

def pad_ticker(x) -> str | None:
    """5930 / 005930 / 005930.0 / 'A005930' 등 -> '005930' (가능한 경우)"""
    if pd.isna(x):
        return None
    s = str(x).strip()

    # 005930.0 같은 케이스 처리
    if s.endswith(".0"):
        s = s[:-2]

    # 숫자만 추출
    digits = re.sub(r"\D", "", s)
    if not digits:
        return s  # 숫자가 없으면 원본 유지(혹시 특수 티커 등)

    # 한국 주식 6자리 기준으로 패딩
    if len(digits) <= 6:
        return digits.zfill(6)

    # 6자리보다 길면 마지막 6자리 쓰는 건 위험할 수 있어 그대로 반환
    return digits

# =========================
# 3) 처리
# =========================
csv_files = sorted(INPUT_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"CSV 파일이 없어요: {INPUT_DIR}")

for fp in csv_files:
    # 인코딩 대응
    try:
        df = pd.read_csv(fp, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(fp, encoding="cp949")

    date_col = pick_col(df, DATE_CANDS)
    tick_col = pick_col(df, TICKER_CANDS)
    pred_col = pick_col(df, PRED_CANDS)
    esg_col  = pick_col(df, ESG_CANDS)

    missing = []
    if date_col is None: missing.append("일자")
    if tick_col is None: missing.append("ticker")
    if pred_col is None: missing.append("pred_label")
    if esg_col is None:  missing.append("ESG_Label")

    if missing:
        print(f"[스킵] {fp.name}: 컬럼 못 찾음 -> {missing} | 현재 컬럼 일부: {list(df.columns)[:15]}")
        continue

    out = df[[date_col, tick_col, pred_col, esg_col]].copy()
    out.columns = ["일자", "ticker", "pred_label", "ESG_Label"]

    # ✅ ticker 0패딩 적용
    out["ticker"] = out["ticker"].apply(pad_ticker)

    out.to_csv(OUT_DIR / fp.name, index=False, encoding="utf-8-sig")
    print(f"[완료] {fp.name} -> {fp.name} (4 cols + ticker pad)")

print(f"✅ 끝! 저장 위치: {OUT_DIR}")


FileNotFoundError: CSV 파일이 없어요: C:\Users\shcho\OneDrive\바탕 화면\최종 프로젝트\추가자료\코드\ESG_ticker분류

In [21]:
pd.read_csv('./ESG_ticker분류_4cols/033530_ESG.csv')

,일자,ticker,pred_label,ESG_Label
0,1970-01-01,33530,1,Environmental
1,1970-01-01,33530,0,Governance
2,1970-01-01,33530,0,Governance
3,1970-01-01,33530,-1,Social
4,1970-01-01,33530,1,Governance
...,...,...,...,...
595,1970-01-01,33530,0,Governance
596,1970-01-01,33530,1,Social
597,1970-01-01,33530,1,Social
598,1970-01-01,33530,0,Environmental
